# Projet Deep Learning for Computer Vision : Figures & Évaluation

**Auteurs :** Olivier BOROT, Marin NAGY  
**Date :** Février 2026

Génère toutes les figures du rapport à partir du modèle entraîné (`yolov5n_custom.pt`) et de `training_history.json`.

---
## 1. Importations et Configuration

In [29]:
import os
import json
import torch
import numpy as np

# PATCH: Patch for NumPy 2.0.0+ (yolov5.utils.metrics uses np.trapz which was removed)
# This restores compatibility with yolov5 on newer numpy versions.
if not hasattr(np, 'trapz'):
    np.trapz = np.trapezoid

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from PIL import Image
from tqdm import tqdm
from pathlib import Path

# Configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

IMG_SIZE = 640
CONF_THRES = 0.25  # Seuil de confiance pour l'inférence
IOU_THRES = 0.45   # Seuil NMS IoU

# Rapport Figures Directory
FIGURES_DIR = '../rapport/figures'
os.makedirs(FIGURES_DIR, exist_ok=True)
print(f"Figures directory: {FIGURES_DIR}")

# Noms des classes (doit matcher l'entraînement)
names = ['ferme', 'immeuble', 'maison', 'piscine', 'usine', 'villa']
print(f"Classes: {names}")

# Chargement du modèle
model_path = 'yolov5n_custom.pt'
if not os.path.exists(model_path):
    print(f"ATTENTION: Le fichier {model_path} est introuvable.")
else:
    print(f"Chargement du modèle depuis {model_path}...")
    # Fix for 'UnpicklingError: Weights only load failed': weights_only=False
    ckpt = torch.load(model_path, map_location=DEVICE, weights_only=False)
    model = ckpt['model'].float().to(DEVICE).eval()
    print("Modèle chargé.")

# Chargement du jeu de test
test_json_path = 'test_images.json'
if os.path.exists(test_json_path):
    with open(test_json_path, 'r') as f:
        test_images = json.load(f)
    print(f"Jeu de test chargé : {len(test_images)} images.")
else:
    print("ATTENTION : 'test_images.json' introuvable.")
    test_images = []

# Utilitaires YOLOv5
from yolov5.utils.general import non_max_suppression, scale_boxes
from yolov5.utils.metrics import box_iou, ap_per_class

Device: cpu
Figures directory: ../rapport/figures
Classes: ['ferme', 'immeuble', 'maison', 'piscine', 'usine', 'villa']
Chargement du modèle depuis yolov5n_custom.pt...
Modèle chargé.
Jeu de test chargé : 15 images.


## 2. Courbe de Perte (Training History)
Évolution de la loss d'entraînement et de validation sur les 100 époques.

In [30]:
history_path = 'training_history.json'

if os.path.exists(history_path):
    with open(history_path, 'r') as f:
        history = json.load(f)

    train_loss = history['train_loss']
    val_loss   = history['val_loss']
    epochs     = list(range(1, len(train_loss) + 1))
    n_epochs   = len(train_loss)

    fig, ax = plt.subplots(figsize=(10, 5))

    ax.plot(epochs, train_loss, color='#2980b9', linewidth=2, label='Train Loss')
    ax.plot(epochs, val_loss,   color='#e74c3c', linewidth=1.5, alpha=0.75, label='Val Loss')

    ax.set_xlabel('Époque', fontsize=12)
    ax.set_ylabel('Loss', fontsize=12)
    ax.set_title(f'Évolution de la perte (Train / Val) sur {n_epochs} époques', fontsize=13)
    ax.legend(fontsize=11)
    ax.grid(alpha=0.3)
    ax.set_ylim(0, max(max(val_loss) * 1.1, 2.0))
    ax.set_xlim(1, n_epochs)

    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'loss_curve.png'), dpi=300)
    plt.show()
    print(f"Sauvegardé : loss_curve.png")
    print(f"Train loss : {train_loss[0]:.3f} (ep1) → {train_loss[-1]:.3f} (ep{n_epochs})")
    print(f"Val loss   : {val_loss[0]:.3f} (ep1) → {val_loss[-1]:.3f} (ep{n_epochs})")
else:
    print(f"ATTENTION : '{history_path}' introuvable.")


Sauvegardé : loss_curve.png
Train loss : 1.760 (ep1) → 0.362 (ep100)
Val loss   : 1.415 (ep1) → 1.021 (ep100)


## 3. Matrice de Confusion
Calcul de la Matrice de Confusion normalisée sur le jeu de test.

In [31]:
n_classes = len(names)
confusion_matrix = np.zeros((n_classes + 1, n_classes + 1))

if len(test_images) > 0 and 'model' in locals():
    print("Calcul de la matrice de confusion...")
    for img_path in tqdm(test_images, desc="Computing Matrix"):
        # Load & Prep Image
        try:
            img0 = Image.open(img_path).convert('RGB')
        except: continue
        
        img = img0.resize((IMG_SIZE, IMG_SIZE))
        img_tensor = torch.from_numpy(np.array(img)).permute(2, 0, 1).float().to(DEVICE) / 255.0
        img_tensor = img_tensor.unsqueeze(0)

        # Load Labels
        label_path = img_path.replace('images', 'labels').replace('.jpg', '.txt').replace('.png', '.txt')
        
        labels_pixel = torch.zeros((0, 5), device=DEVICE)
        
        if os.path.exists(label_path) and os.path.getsize(label_path) > 0:
            l = np.loadtxt(label_path).reshape(-1, 5)
            # Format: class x y w h normalized
            t = torch.from_numpy(l).to(DEVICE)
            # Convert to [class, x1, y1, x2, y2]
            x, y, w, h = t[:, 1]*IMG_SIZE, t[:, 2]*IMG_SIZE, t[:, 3]*IMG_SIZE, t[:, 4]*IMG_SIZE
            x1, y1, x2, y2 = x-w/2, y-h/2, x+w/2, y+h/2
            labels_pixel = torch.stack((t[:, 0], x1, y1, x2, y2), 1)

        # Inference
        with torch.no_grad():
            preds = model(img_tensor)[0]
            preds = non_max_suppression(preds, CONF_THRES, IOU_THRES)
            
        det = preds[0]
        
        # Match Predictions with Ground Truth
        if len(det) == 0:
            # All GT represent False Negatives (Background)
            for t in labels_pixel:
                confusion_matrix[int(t[0]), n_classes] += 1
            continue
            
        # Calc IoU
        iou = box_iou(labels_pixel[:, 1:], det[:, :4])
        # x[0] = target indices, x[1] = detection indices
        x = torch.where(iou > IOU_THRES)
        
        matches = []
        if x[0].shape[0]:
            matches = torch.cat((torch.stack(x, 1), iou[x[0], x[1]][:, None]), 1).cpu().numpy()
            if x[0].shape[0] > 1:
                matches = matches[matches[:, 2].argsort()[::-1]]
                matches = matches[np.unique(matches[:, 1], return_index=True)[1]]
                matches = matches[matches[:, 2].argsort()[::-1]]
                matches = matches[np.unique(matches[:, 0], return_index=True)[1]]
        else:
            matches = np.zeros((0, 3))

        matched_targets = matches[:, 0].astype(int)
        matched_dets = matches[:, 1].astype(int)
        
        # TP (or misclassified object)
        for i, j in zip(matched_targets, matched_dets):
            gt_cls = int(labels_pixel[i, 0])
            pred_cls = int(det[j, 5])
            confusion_matrix[gt_cls, pred_cls] += 1
            
        # FN (Unmatched Targets) -> Background
        all_targets = np.arange(len(labels_pixel))
        unmatched_targets = np.setdiff1d(all_targets, matched_targets)
        for i in unmatched_targets:
            gt_cls = int(labels_pixel[i, 0])
            confusion_matrix[gt_cls, n_classes] += 1 
            
        # FP (Unmatched Detections) -> Background
        all_dets = np.arange(len(det))
        unmatched_dets = np.setdiff1d(all_dets, matched_dets)
        for i in unmatched_dets:
            pred_cls = int(det[i, 5])
            confusion_matrix[n_classes, pred_cls] += 1

    # Plot
    fig, ax = plt.subplots(figsize=(9, 8))
    # Normalize by row (True Class)
    row_sums = confusion_matrix.sum(axis=1)
    cm_norm = confusion_matrix / (row_sums[:, np.newaxis] + 1e-6)

    xticklabels = names + ['Background']
    yticklabels = names + ['Background']

    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=xticklabels, yticklabels=yticklabels)
    plt.ylabel('True Class')
    plt.xlabel('Predicted Class')
    plt.title('Confusion Matrix (Normalized)')
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, "confusion_matrix.png"), dpi=300)
    plt.show()
    print("Sauvegardé : confusion_matrix.png")
else:
    print("Pas d'images ou de modèle pour calculer la matrice.")

Calcul de la matrice de confusion...


Computing Matrix: 100%|██████████| 15/15 [00:01<00:00, 11.64it/s]


Sauvegardé : confusion_matrix.png


## 4. Precision-Recall par classe
Calcul des métriques (P, R, F1, mAP) et affichage des résultats par classe.

In [32]:
def process_batch(detections, labels, iouv):
    """
    Compare les détections aux labels pour calculer les TP (True Positives).
    """
    correct = torch.zeros(detections.shape[0], len(iouv), dtype=torch.bool, device=iouv.device)
    iou = box_iou(labels[:, 1:], detections[:, :4])
    x = torch.where((iou >= iouv[0]) & (labels[:, 0:1] == detections[:, 5]))  # IoU > seuil AND same class
    
    if x[0].shape[0]:
        matches = torch.cat((torch.stack(x, 1), iou[x[0], x[1]][:, None]), 1).cpu().numpy()
        if x[0].shape[0] > 1:
            matches = matches[matches[:, 2].argsort()[::-1]]
            matches = matches[np.unique(matches[:, 1], return_index=True)[1]]
            matches = matches[matches[:, 2].argsort()[::-1]]
            matches = matches[np.unique(matches[:, 0], return_index=True)[1]]
        
        matches = torch.from_numpy(matches).to(iouv.device)
        correct[matches[:, 1].long()] = matches[:, 2:3] >= iouv
        
    return correct

stats = []
iouv = torch.tensor([0.5], device=DEVICE) 

if len(test_images) > 0 and 'model' in locals():
    print("Calcul des métriques P/R...")
    for img_path in tqdm(test_images, desc="Computing Metrics"):
        try:
            img0 = Image.open(img_path).convert('RGB')
        except: continue
        
        img = img0.resize((IMG_SIZE, IMG_SIZE))
        img_tensor = torch.from_numpy(np.array(img)).permute(2, 0, 1).float().to(DEVICE) / 255.0
        img_tensor = img_tensor.unsqueeze(0)
        
        labels_pixel = torch.zeros((0, 5), device=DEVICE)
        label_path = img_path.replace('images', 'labels').replace('.jpg', '.txt').replace('.png', '.txt')
        tcls = []
        if os.path.exists(label_path) and os.path.getsize(label_path) > 0:
            l = np.loadtxt(label_path).reshape(-1, 5)
            t = torch.from_numpy(l).to(DEVICE)
            x, y, w, h = t[:, 1]*IMG_SIZE, t[:, 2]*IMG_SIZE, t[:, 3]*IMG_SIZE, t[:, 4]*IMG_SIZE
            x1, y1, x2, y2 = x-w/2, y-h/2, x+w/2, y+h/2
            labels_pixel = torch.stack((t[:, 0], x1, y1, x2, y2), 1)
            tcls = t[:, 0].tolist()

        with torch.no_grad():
            preds = model(img_tensor)[0]
            preds = non_max_suppression(preds, CONF_THRES, IOU_THRES)
        
        det = preds[0] # (n, 6)
        
        if len(det) == 0:
            if len(tcls):
                stats.append((torch.zeros(0, iouv.numel(), dtype=torch.bool), torch.Tensor(), torch.Tensor(), tcls))
            continue

        if len(labels_pixel):
            correct = process_batch(det, labels_pixel, iouv)
        else:
            correct = torch.zeros(len(det), iouv.numel(), dtype=torch.bool)
            
        stats.append((correct.cpu(), det[:, 4].cpu(), det[:, 5].cpu(), tcls))

    # Compute Global Metrics
    stats = [np.concatenate(x, 0) for x in zip(*stats)]
    if len(stats) and stats[0].any():
        # Fix for 'AttributeError: 'list' object has no attribute 'items''
        names_dict = dict(enumerate(names))
        
        tp, fp, p, r, f1, ap, ap_class = ap_per_class(*stats, plot=False, names=names_dict)
        ap50 = ap[:, 0]
        
        print(f"\n{'Class':<15} {'Images':<10} {'Labels':<10} {'P':<10} {'R':<10} {'F1':<10} {'mAP@.5':<10}")
        for i, c in enumerate(ap_class):
            print(f"{names[c]:<15} {len(test_images):<10} {sum(stats[3]==c):<10} {p[i]:.3f}      {r[i]:.3f}      {f1[i]:.3f}      {ap50[i]:.3f}")
        
        # Plot P/R per class
        x = np.arange(len(p))
        width = 0.35
        fig, ax = plt.subplots(figsize=(10, 6))
        rects1 = ax.bar(x - width/2, p, width, label='Precision', color='#3498db')
        rects2 = ax.bar(x + width/2, r, width, label='Recall', color='#e74c3c')
        ax.set_ylabel('Scores')
        ax.set_title('Precision et Recall par classe (Jeu de Test)')
        ax.set_xticks(x)
        # Use ap_class to align names correctly if some classes missing
        labels = [names[c] for c in ap_class]
        ax.set_xticklabels(labels, rotation=45)
        ax.legend()
        ax.set_ylim([0, 1.1])
        ax.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(FIGURES_DIR, "precision_recall_per_class.png"), dpi=300)
        plt.show()
        print("Sauvegardé : precision_recall_per_class.png")
    else:
        print("Aucune statistique générée (pas de détections).")

Calcul des métriques P/R...


Computing Metrics: 100%|██████████| 15/15 [00:01<00:00, 11.23it/s]



Class           Images     Labels     P          R          F1         mAP@.5    
ferme           15         6          0.000      0.000      0.000      0.000
immeuble        15         174        0.421      0.259      0.320      0.297
maison          15         160        0.473      0.275      0.348      0.329
piscine         15         31         0.591      0.419      0.491      0.522
usine           15         8          0.000      0.000      0.000      0.000
villa           15         3          0.000      0.000      0.000      0.000
Sauvegardé : precision_recall_per_class.png


## 5. Exemples Visuels (Succès & Échecs)
Sélectionne 3 exemples de chaque catégorie pour illustrer les performances.

In [33]:
success_count = 0
failure_count = 0
max_examples = 3

print("Génération des exemples visuels...")

if len(test_images) > 0 and 'model' in locals():
    for img_path in tqdm(test_images):
        if success_count >= max_examples and failure_count >= max_examples:
            break
            
        try:
            img0 = Image.open(img_path).convert('RGB')
        except: continue
        
        img = img0.resize((IMG_SIZE, IMG_SIZE))
        img_tensor = torch.from_numpy(np.array(img)).permute(2, 0, 1).float().to(DEVICE) / 255.0
        img_tensor = img_tensor.unsqueeze(0)

        label_path = img_path.replace('images', 'labels').replace('.jpg', '.txt').replace('.png', '.txt')
        targets = torch.zeros((0, 5), device=DEVICE)
        
        if os.path.exists(label_path) and os.path.getsize(label_path) > 0:
            l = np.loadtxt(label_path).reshape(-1, 5)
            t = torch.from_numpy(l).to(DEVICE)
            x, y, w, h = t[:, 1]*IMG_SIZE, t[:, 2]*IMG_SIZE, t[:, 3]*IMG_SIZE, t[:, 4]*IMG_SIZE
            targets = torch.stack((t[:, 0], x-w/2, y-h/2, x+w/2, y+h/2), 1)

        with torch.no_grad():
            preds = model(img_tensor)[0]
            preds = non_max_suppression(preds, CONF_THRES, IOU_THRES)
            
        det = preds[0]
        
        # Analyze quality
        is_success = False
        is_failure = False
        
        if len(targets) > 0 and len(det) > 0:
            iou = box_iou(targets[:, 1:], det[:, :4])
            # Good match criteria: High IoU and Correct Class
            matches = torch.where((iou > 0.6) & (targets[:, 0:1] == det[:, 5]))[0]
            
            if len(matches) == len(targets) and len(det) == len(targets):
                is_success = True
            elif len(matches) < len(targets) or len(det) > len(targets):
                is_failure = True
                
        elif len(targets) > 0 and len(det) == 0:
            is_failure = True # False Negative
        elif len(targets) == 0 and len(det) > 0:
            is_failure = True # False Positive
            
        # Select & Save
        fname = ""
        if is_success and success_count < max_examples:
            fname = f"prediction_success_{success_count+1}.png"
            success_count += 1
        elif is_failure and failure_count < max_examples:
            fname = f"prediction_failure_{failure_count+1}.png"
            failure_count += 1
        else:
            continue
               
        # Draw Plot
        fig, ax = plt.subplots(1, 1, figsize=(10, 10))
        ax.imshow(img0.resize((IMG_SIZE, IMG_SIZE)))
        ax.axis('off')
        
        # GT (Green)
        for t in targets:
            c, x1, y1, x2, y2 = t
            rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='lime', facecolor='none', linestyle='--')
            ax.add_patch(rect)
            ax.text(x1, y1-5, f"{names[int(c)]} (GT)", color='lime', fontsize=9, fontweight='bold')

        # Pred (Red)
        for *xyxy, conf, cls in det:
            x1, y1, x2, y2 = xyxy
            rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='red', facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, y1-5, f"{names[int(cls)]} {conf:.2f}", color='red', fontsize=9, fontweight='bold')
            
        plt.title(f"{fname} (Vert=GT, Rouge=Pred)")
        plt.tight_layout()
        plt.savefig(os.path.join(FIGURES_DIR, fname))
        plt.close()
        print(f"Sauvegardé : {fname}")

print("Terminé.")

Génération des exemples visuels...


  7%|▋         | 1/15 [00:00<00:07,  1.76it/s]

Sauvegardé : prediction_failure_1.png


 13%|█▎        | 2/15 [00:01<00:09,  1.44it/s]

Sauvegardé : prediction_failure_2.png


 33%|███▎      | 5/15 [00:02<00:04,  2.31it/s]

Sauvegardé : prediction_failure_3.png


100%|██████████| 15/15 [00:03<00:00,  4.38it/s]

Terminé.
